Step 1: Data Preprocessing

In [1]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler,RobustScaler,MaxAbsScaler,StandardScaler,PowerTransformer
from torch.utils.data import DataLoader, TensorDataset

# Set a fixed random seed for reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

def create_9x9_matrix(data_row):
    numeric_row = pd.to_numeric(data_row, errors='coerce').fillna(0)
    num_elements_required = 81
    if len(numeric_row) < num_elements_required:
        numeric_row = np.pad(numeric_row, (0, num_elements_required - len(numeric_row)), 'constant')
    matrix = np.array(numeric_row).reshape(9, 9)
    return matrix

def process_dataset(dataset):
    matrices = []
    for _, row in dataset.iterrows():
        matrix = create_9x9_matrix(row)
        matrices.append(matrix)
    return matrices


# Load dataset
dataset_path = 'random_samples.csv'  # Update this path
df = pd.read_csv(dataset_path)


# Convert columns to numeric, coercing errors will turn 'Infinity' and '-Infinity' into NaN
df['FlowBytes/s'] = pd.to_numeric(df['FlowBytes/s'], errors='coerce')
df['FlowPackets/s'] = pd.to_numeric(df['FlowPackets/s'], errors='coerce')

# Replace infinite values with NaN
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Now, replace NaN in 'Flow Bytes/s' and 'Flow Packets/s' with their respective max value + 1
df['FlowBytes/s'].fillna(df['FlowBytes/s'].max() + 1, inplace=True)
df['FlowPackets/s'].fillna(df['FlowPackets/s'].max() + 1, inplace=True)

# Preprocessing
features = df.iloc[:, :-1].values
labels = df.iloc[:, -1].values

# Normalize the features
scaler = MinMaxScaler()
features = scaler.fit_transform(features)

# Process each row to create 9x9 matrices
matrices = process_dataset(pd.DataFrame(features))

# Convert to a 3D numpy array
X = np.array(matrices)

# Encode the labels
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(labels)

# Split data
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=SEED)

X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.25, random_state=SEED)
# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.long)

Step 2: Building the PDAE Model

In [ ]:
import torch
import torch.nn as nn

KERNEL = 3
HIDDEN = 64   # linear layer width before the conv layer
LATENT = 32   # latent size per branch


class ConvEncoder(nn.Module):
    def __init__(self, n_in, dilation, dropout, latent=LATENT):
        super().__init__()
        conv_len = HIDDEN - dilation * (KERNEL - 1)
        self.net = nn.Sequential(
            nn.Linear(n_in, HIDDEN),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(1, 1, kernel_size=KERNEL, stride=1, dilation=dilation),
            nn.Flatten(),
            nn.Linear(conv_len, latent),
        )

    def forward(self, x):
        return self.net(x.unsqueeze(1))


class ConvDecoder(nn.Module):
    def __init__(self, n_out, dilation, dropout, latent=LATENT):
        super().__init__()
        mid = n_out - dilation * (KERNEL - 1)
        self.net = nn.Sequential(
            nn.Linear(latent, mid),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Unflatten(dim=1, unflattened_size=(1, mid)),
            nn.ConvTranspose1d(1, 1, kernel_size=KERNEL, stride=1, dilation=dilation),
            nn.Flatten(),
            nn.Linear(n_out, n_out),
        )

    def forward(self, z):
        return self.net(z)


class SharedDecoder(nn.Module):
    def __init__(self, n_out, in_dim, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, n_out),
        )

    def forward(self, z):
        return self.net(z)


class DeepAutoencoder(nn.Module):
    def __init__(self, num_input_features, num_classes, dilations=(1,), dropout_rate=0.4):
        super().__init__()
        self.num_input_features = num_input_features
        self.dilations = tuple(dilations)
        self.encoders = nn.ModuleList(
            [ConvEncoder(num_input_features, d, dropout_rate) for d in self.dilations]
        )
        latent_total = LATENT * len(self.dilations)
        if len(self.dilations) == 1:
            self.decoder = ConvDecoder(num_input_features, self.dilations[0], dropout_rate)
        else:
            self.decoder = SharedDecoder(num_input_features, latent_total, dropout_rate)
        self.classifier = nn.Linear(latent_total, num_classes)

    def forward(self, x):
        x = x.view(-1, self.num_input_features)
        z = torch.cat([enc(x) for enc in self.encoders], dim=1)
        return self.decoder(z), self.classifier(z)


# ---- ----
DILATIONS = (1,3)            
num_input_features = 81     
# ---------------------------------------------

num_classes = len(label_encoder.classes_)
print(num_classes)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DeepAutoencoder(num_input_features, num_classes, dilations=DILATIONS).to(device)

15


Step 3: Training the Model

In [7]:
import torch.optim as optim  # Import the optim module

LEARNING_RATE = 0.01  # Adjusted learning rate
BATCH_SIZE = 64     # Adjusted batch size

criterion_reconstruction = nn.L1Loss()
criterion_classification = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

train_data = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
val_data = TensorDataset(X_val_tensor, y_val_tensor)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False)

num_epochs = 100

for epoch in range(num_epochs):
    model.train()
    running_loss_reconstruction = 0.0
    running_loss_classification = 0.0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        decoded, classification = model(inputs.view(-1, num_input_features))
        loss_reconstruction = criterion_reconstruction(decoded, inputs.view(-1, num_input_features))
        loss_classification = criterion_classification(classification, labels)
        loss = loss_reconstruction + loss_classification
        loss.backward()
        optimizer.step()

        running_loss_reconstruction += loss_reconstruction.item()
        running_loss_classification += loss_classification.item()

    print(f'Epoch [{epoch+1}/{num_epochs}], Reconstruction Loss: {running_loss_reconstruction / len(train_loader)}, Classification Loss: {running_loss_classification / len(train_loader)}')

    # Validation
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            decoded, outputs = model(inputs.view(-1, num_input_features)) # فرض بر این است که مدل دو خروجی دارد
            _, predicted = torch.max(outputs, 1) # 'outputs' به جای 'outputs.data' استفاده شده است
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    validation_accuracy = correct / total
    print(f'Epoch {epoch+1}, Validation Accuracy: {validation_accuracy:.4f}')




Epoch [1/100], Reconstruction Loss: 0.03246607070683249, Classification Loss: 0.17468552347327743
Epoch 1, Validation Accuracy: 0.9668
Epoch [2/100], Reconstruction Loss: 0.030971446992090996, Classification Loss: 0.13008181591958004
Epoch 2, Validation Accuracy: 0.9719
Epoch [3/100], Reconstruction Loss: 0.031071243034589118, Classification Loss: 0.11990107978723269
Epoch 3, Validation Accuracy: 0.9733
Epoch [4/100], Reconstruction Loss: 0.031105344750172917, Classification Loss: 0.11030198517230884
Epoch 4, Validation Accuracy: 0.9702
Epoch [5/100], Reconstruction Loss: 0.032508249197585076, Classification Loss: 0.10348508240010894
Epoch 5, Validation Accuracy: 0.9686
Epoch [6/100], Reconstruction Loss: 0.031235693671306814, Classification Loss: 0.10306663801828643
Epoch 6, Validation Accuracy: 0.9746
Epoch [7/100], Reconstruction Loss: 0.031117236333019837, Classification Loss: 0.09988220482998719
Epoch 7, Validation Accuracy: 0.9756
Epoch [8/100], Reconstruction Loss: 0.03110599398

Step 4: Evaluating the Model and Calculating Metrics

In [8]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import csv

model.eval()
true_labels = []
predicted_labels = []

test_data = TensorDataset(X_test_tensor, y_test_tensor)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.view(-1, num_input_features)
        inputs, labels = inputs.to(device), labels.to(device)
        decoded, classification = model(inputs)
        _, predicted = torch.max(classification, 1)
        true_labels.extend(labels.cpu().tolist())
        predicted_labels.extend(predicted.cpu().tolist())

accuracy = accuracy_score(true_labels, predicted_labels)
precision = precision_score(true_labels, predicted_labels, average='weighted', zero_division=1)
recall = recall_score(true_labels, predicted_labels, average='weighted', zero_division=1)
f1 = f1_score(true_labels, predicted_labels, average='weighted', zero_division=1)
conf_matrix = confusion_matrix(true_labels, predicted_labels)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)
print("Confusion Matrix:\n", conf_matrix)

csv_file = "CICIDS2017d3.csv"

# نوشتن داده به فایل CSV
with open(csv_file, mode='w', newline='', encoding='utf-8-sig') as file:
    writer = csv.writer(file)
    writer.writerows(conf_matrix)

print(f"فایل {csv_file} با موفقیت ایجاد شد.")

Accuracy: 0.984900426037848
Precision: 0.9849262048737284
Recall: 0.984900426037848
F1 Score: 0.9838151395041721
Confusion Matrix:
 [[83879     0     2     5     6    40     0     0     1   129     1     0
      0]
 [   83     0     0     0     0     0     0     0     0     0     0     0
      0]
 [   95     0  5002     1     3     0     0     0     0     0     0     0
      0]
 [   20     0     0   369     0     2     0     0     0     0     0     0
      0]
 [  366     0     0     0  6437     0     0     0     0     0     0     0
      0]
 [    8     0     0     0     0   201     3     0     0     1     0     0
      0]
 [   23     0     0     0     0     8   191     4     0     0     0     0
      0]
 [    2     0     0     0     0     0     0   234     0     0     0     0
      0]
 [    0     0     0     0     0     0     0     1     0     0     0     0
      0]
 [  624     0     0     0     3     0     0     0     0  2969     0     0
      0]
 [   10     0     0     0     0     0 